<h3 style="color: #e0e0ff; font-style: italic;">🔍 Baseline Evaluation: BLIP (Bootstrapping Language-Image Pre-training) for Multimodal Fake News Detection</h3>

---

**Research Context & Motivation**
- **Generative vs. Contrastive Encoders**: Standard vision-language pre-trained models like **BLIP** (`Salesforce/blip-image-captioning-base`) are primarily optimized for *natural language generation* (captioning and VQA) rather than classification.
- **Benchmark Objective**: This notebook serves as a comparative baseline to demonstrate that generative VLM architectures (BLIP) are **less effective and computationally heavier** for multimodal fake news detection compared to contrastive alignment models (like **CLIP**).

---

**Model Pipeline Overview**
- **Vision-Language Encoder**: Salesforce BLIP (`blip-image-captioning-base`) frozen feature extractor.
- **Cross-Modal Attention**: Multi-head cross-attention layer to fuse visual patch features and text embeddings.
- **Classifier**: 2-layer MLP with BatchNorm and Dropout for binary classification (Real vs. Fake).


**Imports and Setup**

In [ ]:
# Core Imports
import os
import sys
import time
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Transformers
from transformers import BlipProcessor, BlipForConditionalGeneration

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, accuracy_score, precision_recall_fscore_support, roc_curve, auc
)

import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, "../src")
from data_pipeline import M4FCDataPipeline
from model_registry import update_registry


In [ ]:
# Reproducibility & Device
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {device}")


**Data Loading and Preparation**

In [ ]:
class M4FCBLIPDataset(Dataset):
    def __init__(self, df, processor, max_text_length=128):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_text_length = max_text_length
        self.text_field = 'multilingual_claim'
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_field]) if pd.notna(row[self.text_field]) else ""
        
        image_path = row['full_image_path'] if pd.notna(row['full_image_path']) else ""
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert('RGB')
            else:
                image = Image.new('RGB', (224, 224), color='white')
        except Exception:
            image = Image.new('RGB', (224, 224), color='white')
            
        encoding = self.processor(
            images=image,
            text=text,
            max_length=self.max_text_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        
        label = torch.tensor(row['target'], dtype=torch.long)
        return {
            'pixel_values': encoding['pixel_values'].squeeze(0),
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': label
        }


In [ ]:
print("📂 Loading processor and dataset...")
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
pipeline = M4FCDataPipeline(csv_path='../data/M4FC.csv', target_col='target', random_state=42)
df = pipeline._load_data()

train_df, val_df, test_df = pipeline.split_dataframe(df)
print(f"✓ Dataset split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

train_dataset = M4FCBLIPDataset(train_df, processor)
val_dataset = M4FCBLIPDataset(val_df, processor)
test_dataset = M4FCBLIPDataset(test_df, processor)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


**BLIP Model Architecture**

In [ ]:
class BLIPFakeNewsDetector(nn.Module):
    def __init__(self, num_classes=2, fusion_dim=512, dropout=0.3):
        super().__init__()
        self.blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base", use_safetensors=True)
        self.vision_encoder = self.blip.vision_model
        self.text_encoder = self.blip.text_decoder.bert
        
        # Freeze backbones
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
        for param in self.text_encoder.parameters():
            param.requires_grad = False
            
        vision_dim = self.vision_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size
        
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=fusion_dim,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"✅ BLIP Fake News Detector initialized")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
        
    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values)
            image_features = vision_outputs.last_hidden_state[:, 0, :]
            
            text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            text_features = text_outputs.last_hidden_state[:, 0, :]
            
        proj_image = self.vision_proj(image_features)
        proj_text = self.text_proj(text_features)
        
        image_expanded = proj_image.unsqueeze(1)
        text_expanded = proj_text.unsqueeze(1)
        
        attended_image, _ = self.cross_attention(image_expanded, text_expanded, text_expanded)
        attended_image = attended_image.squeeze(1)
        
        fused = torch.cat([attended_image, proj_text], dim=-1)
        logits = self.classifier(fused)
        return logits


**Training Pipeline**

In [ ]:
class BLIPTrainer:
    def __init__(self, model, train_loader, val_loader, test_loader, device='cuda', patience=10):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.patience = patience
        
        self.optimizer = AdamW(model.parameters(), lr=1e-4)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=20)
        
        num_real = 0
        num_fake = 0
        for batch in self.train_loader:
            labels = batch['label']
            num_real += (labels == 0).sum().item()
            num_fake += (labels == 1).sum().item()
            
        weight_real = 1.0 / num_real if num_real > 0 else 1.0
        weight_fake = 1.0 / num_fake if num_fake > 0 else 1.0
        weights = torch.tensor([weight_real, weight_fake], dtype=torch.float).to(device)
        self.criterion = nn.CrossEntropyLoss(weight=weights)
        
        self.metrics_history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'precision': [], 'recall': [], 'f1': [],
            'auc': []
        }
        
        model_dir = '../models/blip'
        os.makedirs(model_dir, exist_ok=True)
        self.best_model_path = os.path.join(model_dir, 'best_blip_model.pth')
        self.best_val_f1 = 0.0
        self.best_val_acc = 0.0
        
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        predictions = []
        labels = []
        
        progress_bar = tqdm(self.train_loader, desc='Training')
        for batch in progress_bar:
            pixel_values = batch['pixel_values'].to(self.device)
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            label = batch['label'].to(self.device)
            
            self.optimizer.zero_grad()
            logits = self.model(pixel_values, input_ids, attention_mask)
            loss = self.criterion(logits, label)
            
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            _, preds = torch.max(logits, 1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(label.cpu().numpy())
            progress_bar.set_postfix({'loss': loss.item()})
            
        self.scheduler.step()
        avg_loss = total_loss / len(self.train_loader)
        accuracy = accuracy_score(labels, predictions)
        return avg_loss, accuracy

    @torch.no_grad()
    def evaluate(self, data_loader, mode='val'):
        self.model.eval()
        total_loss = 0
        predictions = []
        probabilities = []
        labels = []
        
        for batch in tqdm(data_loader, desc=f'Evaluating {mode}'):
            pixel_values = batch['pixel_values'].to(self.device)
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            label = batch['label'].to(self.device)
            
            logits = self.model(pixel_values, input_ids, attention_mask)
            loss = self.criterion(logits, label)
            
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            _, preds = torch.max(logits, 1)
            
            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
            labels.extend(label.cpu().numpy())
            
        avg_loss = total_loss / len(data_loader)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='binary', zero_division=0
        )
        auc_score = roc_auc_score(labels, probabilities)
        
        return {
            'loss': avg_loss,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc_score,
            'predictions': predictions,
            'true_labels': labels,
            'probabilities': probabilities
        }

    def train(self, epochs=10):
        print(f"Starting BLIP Training Pipeline for {epochs} epochs...")
        no_improve_count = 0
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss, train_acc = self.train_epoch()
            val_metrics = self.evaluate(self.val_loader, mode='val')
            
            self.metrics_history['train_loss'].append(train_loss)
            self.metrics_history['train_acc'].append(train_acc)
            self.metrics_history['val_loss'].append(val_metrics['loss'])
            self.metrics_history['val_acc'].append(val_metrics['accuracy'])
            self.metrics_history['precision'].append(val_metrics['precision'])
            self.metrics_history['recall'].append(val_metrics['recall'])
            self.metrics_history['f1'].append(val_metrics['f1'])
            self.metrics_history['auc'].append(val_metrics['auc'])
            
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
            print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Acc:   {val_metrics['accuracy']:.4f} | Val F1: {val_metrics['f1']:.4f} | Val AUC: {val_metrics['auc']:.4f}")
            
            if val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                self.best_val_acc = val_metrics['accuracy']
                no_improve_count = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_f1': val_metrics['f1'],
                    'val_acc': val_metrics['accuracy']
                }, self.best_model_path)
                print(f"  🏆 Best model saved with Val F1={val_metrics['f1']:.4f} to {self.best_model_path}")
            else:
                no_improve_count += 1
                if no_improve_count >= self.patience:
                    print(f"Early stopping triggered after {epoch+1} epochs.")
                    break
                    
        return self.metrics_history

    @torch.no_grad()
    def test(self):
        print("\nEvaluating best BLIP model on test set...")
        checkpoint = torch.load(self.best_model_path, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        start_time = time.time()
        test_metrics = self.evaluate(self.test_loader, mode='test')
        inference_time = time.time() - start_time
        test_metrics['inference_time'] = inference_time
        
        print("\nTest Set Results:")
        print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
        print(f"Precision: {test_metrics['precision']:.4f}")
        print(f"Recall:    {test_metrics['recall']:.4f}")
        print(f"F1-Score:  {test_metrics['f1']:.4f}")
        print(f"AUC:       {test_metrics['auc']:.4f}")
        print(f"⏱️ Inference time: {inference_time:.2f} seconds")
        return test_metrics


**Run Training & Evaluation**

In [ ]:
model = BLIPFakeNewsDetector().to(device)
trainer = BLIPTrainer(model, train_loader, val_loader, test_loader, device=device, patience=5)

metrics_history = trainer.train(epochs=10)
test_metrics = trainer.test()


**Visualization Suite**

In [ ]:
class BLIPVisualizer:
    @staticmethod
    def plot_all(metrics_history, test_metrics, save_dir='../reports/blip/figures'):
        os.makedirs(save_dir, exist_ok=True)
        
        # Training history
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes[0, 0].plot(metrics_history['train_loss'], label='Train Loss')
        axes[0, 0].plot(metrics_history['val_loss'], label='Val Loss')
        axes[0, 0].set_title('Loss Over Time')
        axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(metrics_history['train_acc'], label='Train Acc')
        axes[0, 1].plot(metrics_history['val_acc'], label='Val Acc')
        axes[0, 1].set_title('Accuracy Over Time')
        axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(metrics_history['f1'], label='F1-Score', marker='o')
        axes[1, 0].plot(metrics_history['auc'], label='AUC', marker='s')
        axes[1, 0].set_title('F1-Score and AUC')
        axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(metrics_history['precision'], label='Precision', marker='^')
        axes[1, 1].plot(metrics_history['recall'], label='Recall', marker='v')
        axes[1, 1].set_title('Precision-Recall Trade-off')
        axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        history_path = os.path.join(save_dir, 'blip_training_history.png')
        plt.savefig(history_path, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"✅ Training history plot saved to {history_path}")
        
        # Confusion Matrix
        cm = confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
        plt.title('Confusion Matrix - BLIP Model')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        cm_path = os.path.join(save_dir, 'blip_confusion_matrix.png')
        plt.savefig(cm_path, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"✅ Confusion matrix saved to {cm_path}")
        
        # ROC Curve
        fpr, tpr, _ = roc_curve(test_metrics['true_labels'], test_metrics['probabilities'])
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(7, 5))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.title('ROC Curve - BLIP')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        roc_path = os.path.join(save_dir, 'blip_roc_curve.png')
        plt.savefig(roc_path, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"✅ ROC curve saved to {roc_path}")

BLIPVisualizer.plot_all(metrics_history, test_metrics)


**Reporting & Registry Update**

In [ ]:
class BLIPModelSaver:
    @staticmethod
    def generate_report(metrics_history, test_metrics, save_name="BLIP_Standard"):
        report_dir = "../reports/blip/metrics"
        os.makedirs(report_dir, exist_ok=True)
        report_path = os.path.join(report_dir, f"{save_name}_Report.txt")
        
        y_true = test_metrics['true_labels']
        y_pred = test_metrics['predictions']
        
        p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
        
        best_val_acc = max(metrics_history['val_acc']) if metrics_history['val_acc'] else test_metrics['accuracy']
        best_val_f1 = max(metrics_history['f1']) if metrics_history['f1'] else test_metrics['f1']
        
        report_content = f"""==================================================
{save_name} MODEL FAKE NEWS DETECTION REPORT
==================================================
Test Accuracy:         {test_metrics['accuracy']:.6f}
Test Precision (Bin):  {test_metrics['precision']:.6f}
Test Recall (Bin):     {test_metrics['recall']:.6f}
Test F1-Score (Bin):   {test_metrics['f1']:.6f}
Test F1-Score (Macro): {f1_macro:.6f}
Test F1-Score (Wtd):   {f1_weighted:.6f}
Test AUC-ROC:          {test_metrics['auc']:.6f}
Inference Time (sec):  {test_metrics['inference_time']:.2f}
==================================================
"""
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report_content)
            
        print(f"✅ Performance report saved to {report_path}")
        
        # Update registry safely
        update_registry(
            name="BLIP",
            arch="BLIPFakeNewsDetector",
            rel_path="models/blip/best_blip_model.pth",
            val_accuracy=float(best_val_acc),
            val_f1=float(best_val_f1),
            test_accuracy=float(test_metrics['accuracy']),
            test_f1=float(test_metrics['f1']),
            test_auc=float(test_metrics['auc']),
            img_dim=768,
            text_dim=768,
            num_classes=2
        )

BLIPModelSaver.generate_report(metrics_history, test_metrics)
